In [1]:
import torch
import torchvision
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

PyTorch version: 2.14.0+cpu
Torchvision version: 0.29.0+cpu


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cpu")

In [3]:
transform = transforms.Compose([transforms.ToTensor()])
test_set = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
test_loader = DataLoader(test_set, batch_size=1000, shuffle=False)

100%|██████████| 9.91M/9.91M [00:12<00:00, 809kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 105kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 790kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.64MB/s]


In [4]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [5]:
model = LeNet().to(device)

In [6]:
train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [8]:
model.train()
for epoch in range(2):
    for x, y in train_loader:
        optimizer.zero_grad()
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss.backward()
        optimizer.step()
    print(f"epoch {epoch} done, loss={loss.item():.4f}")

epoch 0 done, loss=0.0656
epoch 1 done, loss=0.2044


In [9]:
torch.save(model.state_dict(), "lenet_mnist.pt")

In [10]:
def evaluate(model):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            out = model(x)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
    return correct / len(test_set)

In [11]:
exact_acc = evaluate(model)
print("Exact accuracy:", exact_acc)

Exact accuracy: 0.9809


In [12]:
import numpy as np

def truncated_multiply(a, b):
    exact = a.astype(np.int64) * b.astype(np.int64)
    return exact & ~15  # zero lowest 4 bits of product, same as truncated_multiplier.v

def leading_one_pos(x):
    x = x.astype(np.int64)
    valid = x > 0
    pos = np.where((x & 0x80) != 0, 7,
          np.where((x & 0x40) != 0, 6,
          np.where((x & 0x20) != 0, 5,
          np.where((x & 0x10) != 0, 4,
          np.where((x & 0x08) != 0, 3,
          np.where((x & 0x04) != 0, 2,
          np.where((x & 0x02) != 0, 1, 0)))))))
    return pos, valid

def mitchell_multiply(a, b):
    a = a.astype(np.int64); b = b.astype(np.int64)
    pos_a, valid_a = leading_one_pos(a)
    pos_b, valid_b = leading_one_pos(b)

    def mantissa(x, pos):
        below = x & ((1 << pos) - 1)
        shift = 7 - pos
        return (below * (2 ** shift)).astype(np.int64)

    mant_a = mantissa(a, pos_a)
    mant_b = mantissa(b, pos_b)

    mantissa_sum = mant_a + mant_b
    carry = (mantissa_sum > 127).astype(np.int64)
    sum_mantissa = np.where(carry == 1, mantissa_sum - 128, mantissa_sum)
    sum_char = pos_a + pos_b + carry

    significand = 128 + sum_mantissa
    shift = sum_char - 7
    result = np.where(shift >= 0,
                       significand * (2 ** np.clip(shift, 0, 20)),
                       significand // (2 ** np.clip(-shift, 0, 20)))

    return np.where(valid_a & valid_b, result, 0).astype(np.int64)

def approx_multiply_signed(a_int, b_int, method):
    a_sign = (a_int < 0).astype(np.int64)
    b_sign = (b_int < 0).astype(np.int64)
    a_mag = np.abs(a_int); b_mag = np.abs(b_int)

    if method == "truncated":
        mag = truncated_multiply(a_mag, b_mag)
    elif method == "mitchell":
        mag = mitchell_multiply(a_mag, b_mag)
    else:
        mag = a_mag * b_mag

    sign = a_sign ^ b_sign
    return np.where(sign == 1, -mag, mag)

In [13]:
def quantize_tensor(t, bits=8):
    t_np = t.detach().cpu().numpy()
    max_abs = np.max(np.abs(t_np))
    if max_abs == 0:
        max_abs = 1e-8
    scale = max_abs / (2**(bits-1) - 1)
    q = np.round(t_np / scale).astype(np.int64)
    q = np.clip(q, -(2**(bits-1)-1), 2**(bits-1)-1)
    return q, scale

In [14]:
def approx_conv2d(x, weight, bias, stride, padding, method):
    x_q, x_scale = quantize_tensor(x)
    w_q, w_scale = quantize_tensor(weight)

    Cout, Cin, kh, kw = weight.shape
    x_t = torch.from_numpy(x_q).float()
    unfolded = F.unfold(x_t, kernel_size=(kh, kw), padding=padding, stride=stride)
    unfolded = unfolded.numpy().astype(np.int64)  # [N, Cin*kh*kw, L]

    w_flat = w_q.reshape(Cout, -1)  # [Cout, Cin*kh*kw]
    N, K, L = unfolded.shape
    out = np.zeros((N, Cout, L), dtype=np.int64)

    for n in range(N):
        a = unfolded[n]  # [K, L]
        for c in range(Cout):
            wv = w_flat[c][:, None]
            prod = approx_multiply_signed(np.broadcast_to(wv, (K, L)), a, method)
            out[n, c, :] = prod.sum(axis=0)

    out_h = (x.shape[2] + 2*padding - kh) // stride + 1
    out_w = (x.shape[3] + 2*padding - kw) // stride + 1
    out = out.reshape(N, Cout, out_h, out_w)

    out_t = torch.from_numpy(out.astype(np.float64) * x_scale * w_scale).float()
    if bias is not None:
        out_t = out_t + bias.view(1, -1, 1, 1)
    return out_t

In [15]:
def forward_approx(model, x, method):
    x = approx_conv2d(x, model.conv1.weight, model.conv1.bias, stride=1, padding=2, method=method)
    x = F.max_pool2d(F.relu(x), 2)
    x = approx_conv2d(x, model.conv2.weight, model.conv2.bias, stride=1, padding=0, method=method)
    x = F.max_pool2d(F.relu(x), 2)
    x = x.view(x.size(0), -1)
    x = F.relu(model.fc1(x))
    x = F.relu(model.fc2(x))
    return model.fc3(x)

In [16]:
test_subset = torch.utils.data.Subset(test_set, range(200))
test_loader_small = DataLoader(test_subset, batch_size=50, shuffle=False)

In [17]:
def evaluate_approx(model, method):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader_small:
            out = forward_approx(model, x, method)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

In [18]:
exact_subset_acc = evaluate_approx(model, "exact")
trunc_acc = evaluate_approx(model, "truncated")
mitchell_acc = evaluate_approx(model, "mitchell")

print("Exact (subset):     ", exact_subset_acc)
print("Truncated multiplier:", trunc_acc)
print("Mitchell multiplier: ", mitchell_acc)

Exact (subset):      0.995
Truncated multiplier: 0.995
Mitchell multiplier:  0.995


In [19]:
x_sample, y_sample = next(iter(test_loader_small))
x_one = x_sample[:1]

with torch.no_grad():
    out_exact = forward_approx(model, x_one, "exact")
    out_trunc = forward_approx(model, x_one, "truncated")
    out_mitch = forward_approx(model, x_one, "mitchell")

print("Exact logits:    ", out_exact.numpy().round(3))
print("Truncated logits:", out_trunc.numpy().round(3))
print("Mitchell logits: ", out_mitch.numpy().round(3))
print()
print("Max abs diff (truncated vs exact):", (out_exact - out_trunc).abs().max().item())
print("Max abs diff (mitchell vs exact): ", (out_exact - out_mitch).abs().max().item())

Exact logits:     [[ -4.352  -1.701  -1.321  -0.13   -6.359  -2.961 -17.739  11.656  -2.704
    1.918]]
Truncated logits: [[ -4.341  -1.694  -1.31   -0.138  -6.341  -2.961 -17.699  11.639  -2.705
    1.909]]
Mitchell logits:  [[ -4.089  -1.569  -1.24   -0.095  -6.014  -2.713 -16.675  10.95   -2.512
    1.731]]

Max abs diff (truncated vs exact): 0.04048728942871094
Max abs diff (mitchell vs exact):  1.0642948150634766
